In [28]:
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 1. 设定参数
SEQ_LENGTH = 50
CORE_MOTIF = "UACGUAUGCA" # 长度变成 10
NUM_SAMPLES = 4000    # 增加数据量让 AI 充分学习统计分布
MUTATION_RATE = 0.15  # 15% 的突变率足以制造干扰，但不至于摧毁信号

def generate_fuzzy_motif(motif, mutation_rate):
    """模拟真实的生物学 Motif：允许一定概率的碱基突变"""
    bases = ['A', 'U', 'C', 'G']
    mutated_motif = ""
    for base in motif:
        if random.random() < mutation_rate:
            # 发生突变：随机换成另外三个碱基中的一个
            mutated_motif += random.choice([b for b in bases if b != base])
        else:
            # 保持保守序列
            mutated_motif += base
    return mutated_motif

def generate_rna_fuzzy(has_motif=False):
    """随机生成一条 RNA 序列"""
    bases = ['A', 'U', 'C', 'G']
    seq = ''.join(random.choices(bases, k=SEQ_LENGTH))

    if has_motif:
        # 阳性样本：插入经过“突变洗礼”的模糊 Motif
        fuzzy_motif = generate_fuzzy_motif(CORE_MOTIF, MUTATION_RATE)
        insert_pos = random.randint(0, SEQ_LENGTH - len(CORE_MOTIF))
        seq = seq[:insert_pos] + fuzzy_motif + seq[insert_pos + len(CORE_MOTIF):]
    return seq

# 2. 生成更贴近真实生物学的数据集
data = []
for _ in range(NUM_SAMPLES):
    data.append({'sequence': generate_rna_fuzzy(has_motif=True), 'label': 1})
for _ in range(NUM_SAMPLES):
    data.append({'sequence': generate_rna_fuzzy(has_motif=False), 'label': 0})

df = pd.DataFrame(data)
df = df.sample(frac=1).reset_index(drop=True)

print("升级版数据集（包含简并 Motif）前5行预览：")
print(df.head())

升级版数据集（包含简并 Motif）前5行预览：
                                            sequence  label
0  CAGUCCCGCGUACACGUAAUUACGGAAUGGAUGCAGGCAGGGGUGC...      0
1  UAUUUUAUCAUACCCCCAUAUGUAAACGUAGGCACUUGCAUCCACA...      1
2  UACAUCUGCAAUACACCGGCCAAUCCAAAGCCUAGUGCUAUAAAUG...      1
3  AAGUAUACACCGUCGCGAUGGUAGAAAACCUUUCCUCAGUUGGAUG...      0
4  ACCACUUGUUGGGCAAACCUCCCGGUACCCUGAGAAAGUGCAACAC...      0


In [29]:
def sequence_to_onehot(seq):
    """
    将 RNA 序列转化为数字矩阵。
    A -> [1,0,0,0], U -> [0,1,0,0], C -> [0,0,1,0], G -> [0,0,0,1]
    """
    mapping = {
        'A': [1, 0, 0, 0],
        'U': [0, 1, 0, 0],
        'C': [0, 0, 1, 0],
        'G': [0, 0, 0, 1]
    }
    # 把序列里的每个碱基变成4个数字，然后展平变成一个长一维数组 (50*4 = 200个数字)
    onehot = []
    for base in seq:
        onehot.extend(mapping[base])
    return np.array(onehot, dtype=np.float32)

# 把 Pandas 里的序列列全部转化为矩阵
X = np.array([sequence_to_onehot(seq) for seq in df['sequence']])
y = df['label'].values.astype(np.float32)

print(f"X (特征) 的形状: {X.shape}") # 预期: (2000, 200)，即2000条数据，每条200个特征
print(f"y (标签) 的形状: {y.shape}") # 预期: (2000,)

X (特征) 的形状: (8000, 200)
y (标签) 的形状: (8000,)


In [30]:
class RNADataset(Dataset):
    """定义一个标准的 PyTorch 数据集类"""
    def __init__(self, features, labels):
        # 把 Numpy 数组转化为 PyTorch 专属的 Tensor (张量)
        self.X = torch.tensor(features)
        self.y = torch.tensor(labels).unsqueeze(1) # 调整标签形状以匹配模型输出

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 划分训练集 (80%) 和测试集 (20%)
train_size = int(0.8 * len(df))
train_dataset = RNADataset(X[:train_size], y[:train_size])
test_dataset = RNADataset(X[train_size:], y[train_size:])

# DataLoader 负责按批次把数据喂给模型，batch_size=32 意味着每次学习32条序列
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [31]:
class MotifClassifierCNN(nn.Module):
    def __init__(self):
        super(MotifClassifierCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=4, out_channels=16, kernel_size=10)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)

        # 【核心升级】：加入 Dropout 层。
        # 训练时，每次随机蒙住 50% 的神经元，强迫剩下的神经元去学习真正的 Motif 规律，防止死记硬背！
        self.dropout = nn.Dropout(p=0.5)

        self.fc = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.view(-1, 50, 4)
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)

        x = self.dropout(x) # 在进入最后打分前，进行随机干扰

        x = self.fc(x)
        return self.sigmoid(x)

model = MotifClassifierCNN()

In [32]:
# 1. 定义损失函数 (Loss) 和优化器 (Optimizer)
ccriterion = nn.BCELoss()
# 【核心升级】：调低学习率，增加训练轮数
optimizer = optim.Adam(model.parameters(), lr=0.002)

EPOCHS = 30 # 对于带有噪音的数据，15 轮不够，需要给 AI 更多时间

print("🚀 开启最终抗噪训练...")

for epoch in range(EPOCHS):
    model.train() # 开启训练模式
    total_loss = 0

    # 每次从 DataLoader 里拿取 32 条数据 (batch_X) 和真实标签 (batch_y)
    for batch_X, batch_y in train_loader:
        # Step 1: 清空前一步的梯度（必须做）
        optimizer.zero_grad()

        # Step 2: 让模型去猜（前向传播）
        predictions = model(batch_X)

        # Step 3: 计算模型猜的与真实结果之间的差距（算误差）
        loss = criterion(predictions, batch_y)

        # Step 4: 把误差反向传回去（反向传播）
        loss.backward()

        # Step 5: 优化器根据误差调整内部参数
        optimizer.step()

        total_loss += loss.item()

    # --- 每训练完一轮，在测试集上看看效果 ---
    model.eval() # 开启评估模式
    correct = 0
    total = 0
    with torch.no_grad(): # 评估时不需要计算梯度，省内存
        for test_X, test_y in test_loader:
            outputs = model(test_X)
            # 如果输出概率 > 0.5，我们就认为它预测是阳性 (1)
            predicted = (outputs > 0.5).float()
            total += test_y.size(0)
            correct += (predicted == test_y).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], 训练误差 Loss: {total_loss/len(train_loader):.4f}, 测试集准确率: {accuracy:.2f}%")

print("🎉 训练完成！如果准确率接近 100%，说明模型成功发现了 Motif 的规律。")

🚀 开启最终抗噪训练...
Epoch [1/30], 训练误差 Loss: 0.6542, 测试集准确率: 88.75%
Epoch [2/30], 训练误差 Loss: 0.4609, 测试集准确率: 91.31%
Epoch [3/30], 训练误差 Loss: 0.3797, 测试集准确率: 91.69%
Epoch [4/30], 训练误差 Loss: 0.3433, 测试集准确率: 92.31%
Epoch [5/30], 训练误差 Loss: 0.3232, 测试集准确率: 91.94%
Epoch [6/30], 训练误差 Loss: 0.3151, 测试集准确率: 92.44%
Epoch [7/30], 训练误差 Loss: 0.3178, 测试集准确率: 92.31%
Epoch [8/30], 训练误差 Loss: 0.2960, 测试集准确率: 92.25%
Epoch [9/30], 训练误差 Loss: 0.3024, 测试集准确率: 92.25%
Epoch [10/30], 训练误差 Loss: 0.3048, 测试集准确率: 92.44%
Epoch [11/30], 训练误差 Loss: 0.2973, 测试集准确率: 91.94%
Epoch [12/30], 训练误差 Loss: 0.2912, 测试集准确率: 92.38%
Epoch [13/30], 训练误差 Loss: 0.3011, 测试集准确率: 91.88%
Epoch [14/30], 训练误差 Loss: 0.2923, 测试集准确率: 92.00%
Epoch [15/30], 训练误差 Loss: 0.2884, 测试集准确率: 92.31%
Epoch [16/30], 训练误差 Loss: 0.2880, 测试集准确率: 92.31%
Epoch [17/30], 训练误差 Loss: 0.2803, 测试集准确率: 92.25%
Epoch [18/30], 训练误差 Loss: 0.2849, 测试集准确率: 92.50%
Epoch [19/30], 训练误差 Loss: 0.2821, 测试集准确率: 92.12%
Epoch [20/30], 训练误差 Loss: 0.2833, 测试集准确率: 92.31%
Epoch [21/30], 

In [33]:
import torch
import numpy as np

# 1. 确保模型处于“评估模式”（关闭 Dropout 等训练专属功能）
model.eval()

# 2. 定义预测流水线
def predict_new_rna(sequence_list, model):
    """
    接收一个未知的 RNA 序列列表，输出模型预测的打分。
    """
    results = []

    for seq in sequence_list:
        # 【极其关键的预处理】：真实序列长短不一，我们的模型只吃 50bp。
        # 这里做一个简单的截断或补齐 (Padding)
        if len(seq) > 50:
            seq = seq[:50] # 太长就截断
        elif len(seq) < 50:
            seq = seq + 'A' * (50 - len(seq)) # 太短就用A补齐（真实科研中会用 N 即全0矩阵补齐）

        # 转化成数字矩阵
        seq_matrix = sequence_to_onehot(seq)

        # 转化为 PyTorch 张量，并增加一个 Batch 维度 (变成 1x200)
        tensor_x = torch.tensor(seq_matrix).unsqueeze(0)

        # 不要计算梯度，节省算力
        with torch.no_grad():
            score = model(tensor_x).item() # 吐出 0~1 的概率分

        results.append({
            'Sequence': seq,
            'Score': round(score, 4),
            'Prediction': 'Positive (Functional)' if score > 0.5 else 'Negative (Background)'
        })

    return pd.DataFrame(results)

# ==========================================
# 🚀 见证实用的高光时刻：开始虚拟实验！
# ==========================================

# 场景 1：虚拟高通量筛选 (输入从实验室刚测序出来的 3 条未知序列)
print("--- 场景 1: 虚拟筛选 (Virtual Screening) ---")
unknown_sequences = [
    "CGAUGCUACGUAUGCAUUAGCGAUGCGAUCGAUGCUAGCUAGCUAGCUAG",  # 我偷偷把 Motif "UACGUAUGCA" 塞进去了
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA",  # 一段无意义的 poly-A 尾巴
    "CGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCG"   # 一段富含 GC 的背景序列
]
predictions_df = predict_new_rna(unknown_sequences, model)
print(predictions_df.to_markdown(index=False))
print("\n")


# 场景 2：In-silico 虚拟突变实验 (测试某个 SNP 突变是否致命)
print("--- 场景 2: 虚拟突变实验 (In-silico Mutagenesis) ---")
# 这是一条天然带有完美 Motif 的序列，原本功能很强
wild_type_seq = "CGAUGCUACGUAUGCAUUAGCGAUGCGAUCGAUGCUAGCUAGCUAGCUAG"

# 我们在实验室里发现了一个突变株，仅仅突变了中间的两个碱基 (UACGUAUGCA -> UACGCCCGCA)
mutated_seq   = "CGAUGCUACGCCCGCAUUAGCGAUGCGAUCGAUGCUAGCUAGCUAGCUAG"

mutation_test = predict_new_rna([wild_type_seq, mutated_seq], model)
# 给表格加个标签方便看
mutation_test.insert(0, 'Type', ['野生型 (Wild-type)', '突变型 (Mutated)'])
print(mutation_test.to_markdown(index=False))

--- 场景 1: 虚拟筛选 (Virtual Screening) ---
| Sequence                                           |   Score | Prediction            |
|:---------------------------------------------------|--------:|:----------------------|
| CGAUGCUACGUAUGCAUUAGCGAUGCGAUCGAUGCUAGCUAGCUAGCUAG |  1      | Positive (Functional) |
| AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA |  0.1778 | Negative (Background) |
| CGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCG |  0.165  | Negative (Background) |


--- 场景 2: 虚拟突变实验 (In-silico Mutagenesis) ---
| Type               | Sequence                                           |   Score | Prediction            |
|:-------------------|:---------------------------------------------------|--------:|:----------------------|
| 野生型 (Wild-type) | CGAUGCUACGUAUGCAUUAGCGAUGCGAUCGAUGCUAGCUAGCUAGCUAG |  1      | Positive (Functional) |
| 突变型 (Mutated)   | CGAUGCUACGCCCGCAUUAGCGAUGCGAUCGAUGCUAGCUAGCUAGCUAG |  0.2349 | Negative (Background) |
